# Baseline: Supervised Encoder Without Pretraining

This notebook trains the existing DME encoder architecture from random initialization on the downstream supervised task only. It runs the same fractions/seeds protocol and reports per-seed, mean, and std rows.

In [ ]:
# Cell 1 - Setup
import json
import pathlib
import pickle
import random
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "/kaggle/working/denoising-event-sequences", "--quiet"],
    check=True,
)
sys.path.insert(0, "/kaggle/working/denoising-event-sequences")

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader

from src.data.collate import collate_fn
from src.data.dataset import EventSequenceDataset
from src.data.preprocessing import EventPreprocessor
from src.data.splits import make_entity_splits
from src.models.dme_encoder import DMEEncoder
from src.training.finetune import evaluate_finetune, finetune
from src.utils.logging import MetricsLogger
from src.utils.seed import set_seed

SMOKE_RUN = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("Device:", device)


In [ ]:
# Cell 2 - Paths and config
WORKING_DIR = pathlib.Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data"
OUTPUT_DIR = WORKING_DIR / "outputs"
CKPT_DIR = OUTPUT_DIR / "checkpoints_supervised_encoder"
LOG_DIR = OUTPUT_DIR / "logs_supervised_encoder"
for p in [OUTPUT_DIR, CKPT_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

config = {
    "data": {
        "max_seq_len": 256,
        "min_seq_len": 2,
        "event_type_col": "MCC",
        "timestamp_col": "TRDATETIME",
        "target_col": "target_flag",
        "numerical_cols": ["amount"],
        "categorical_cols": ["channel_type", "trx_category"],
        "group_col": "cl_id",
        "truncation_pretrain": "sliding_window",
        "truncation_eval": "last_events",
        "amount_transform": "robust_scaler",
        "time_transform": "none",
        "train_ratio": 0.70,
        "val_ratio": 0.15,
        "test_ratio": 0.15,
        "min_vocab_count": 5,
    },
    "model": {
        "event_type_emb_dim": 64,
        "cat_emb_dim": 32,
        "num_projection_dim": 64,
        "time_projection_dim": 64,
        "hidden_dim": 256,
        "num_layers": 4,
        "num_heads": 8,
        "dim_feedforward": 1024,
        "dropout": 0.10,
        "activation": "gelu",
        "max_seq_len": 256,
    },
    "pooling": {"type": "gated_attention"},
    "training": {
        "batch_size": 128,
        "num_epochs_finetune": 20,
        "lr": 3e-4,
        "lr_encoder": 3e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.05,
        "gradient_clip_val": 1.0,
        "mixed_precision": USE_AMP,
        "early_stopping_patience": 5,
        "log_every_n_steps": 50,
    },
}

LABEL_FRACTIONS = [0.05, 0.25, 1.00]
LABEL_SAMPLING_SEEDS = [42, 43, 44]
METRIC_COLS = ["accuracy", "macro_f1", "weighted_f1", "roc_auc", "pr_auc", "balanced_accuracy"]

if SMOKE_RUN:
    config["training"]["num_epochs_finetune"] = 2
    config["training"]["early_stopping_patience"] = 1
print("Output dir:", OUTPUT_DIR)

In [ ]:
# Cell 3 - Load data, preprocessor, and shared splits
GROUP_COL = config["data"]["group_col"]
TARGET_COL = config["data"]["target_col"]
TIME_COL = config["data"]["timestamp_col"]

df = pd.read_csv(DATA_DIR / "rosbank" / "train.csv.gz")
df[TIME_COL] = pd.to_datetime(df[TIME_COL], format="%d%b%y:%H:%M:%S")
labels_by_entity = df.groupby(GROUP_COL)[TARGET_COL].first().to_dict()

splits = make_entity_splits(
    df,
    entity_col=GROUP_COL,
    target_col=TARGET_COL,
    train_ratio=config["data"]["train_ratio"],
    val_ratio=config["data"]["val_ratio"],
    test_ratio=config["data"]["test_ratio"],
    seed=42,
)

prep = EventPreprocessor(config)
prep.fit(df, splits["train"])

vocab_info = {
    "event_type_vocab_size": len(prep.vocab[prep.event_type_col]),
    "cat_vocab_sizes": [len(prep.vocab[c]) for c in prep.categorical_cols],
    "num_num_features": len(prep.numerical_cols),
    "num_classes": 2,
}

with open(CKPT_DIR / "preprocessor.pkl", "wb") as f:
    pickle.dump(prep, f)
(CKPT_DIR / "splits.json").write_text(json.dumps(splits, indent=2))

print(df.shape)
print({k: len(v) for k, v in splits.items()})
print(vocab_info)


In [ ]:
# Cell 4 - Shared sampling, metrics, and aggregation helpers
def get_entity_labels(df, group_col, target_col):
    return df.groupby(group_col)[target_col].first().to_dict()


def sample_labeled_entities(train_ids, labels_by_entity, fraction, seed):
    train_ids = list(train_ids)
    if fraction >= 1.0:
        return train_ids

    rng = np.random.default_rng(seed)
    by_label = {}
    for entity_id in train_ids:
        label = int(labels_by_entity[entity_id])
        by_label.setdefault(label, []).append(entity_id)

    sampled = []
    for label, ids in sorted(by_label.items()):
        ids = np.array(ids, dtype=object)
        n = max(1, int(round(len(ids) * fraction)))
        n = min(n, len(ids))
        sampled.extend(rng.choice(ids, size=n, replace=False).tolist())

    rng.shuffle(sampled)
    return sampled


def compute_binary_metrics(y_true, pos_proba):
    y_true = np.asarray(y_true).astype(int)
    pos_proba = np.asarray(pos_proba).astype(float)
    y_pred = (pos_proba >= 0.5).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, pos_proba)),
        "pr_auc": float(average_precision_score(y_true, pos_proba)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


def append_mean_std_rows(rows, fractions, metric_cols=METRIC_COLS):
    final_rows = []
    for fraction in fractions:
        runs = [r for r in rows if float(r["fraction"]) == float(fraction)]
        final_rows.extend(runs)
        mean_row = {"fraction": fraction, "seed": "mean", "n_seeds": len(runs)}
        std_row = {"fraction": fraction, "seed": "std", "n_seeds": len(runs)}
        for metric in metric_cols:
            vals = [float(r[metric]) for r in runs if metric in r and pd.notna(r[metric])]
            if vals:
                mean_row[metric] = float(np.mean(vals))
                std_row[metric] = float(np.std(vals, ddof=0))
        final_rows.append(mean_row)
        final_rows.append(std_row)
    return final_rows


def save_results(rows, stem):
    results_df = pd.DataFrame(rows)
    csv_path = OUTPUT_DIR / f"{stem}.csv"
    json_path = OUTPUT_DIR / f"{stem}.json"
    results_df.to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(rows, indent=2, ensure_ascii=False))
    print(f"Saved: {csv_path}")
    print(f"Saved: {json_path}")
    return results_df


In [ ]:
# Cell 5 - Shared validation/test loaders and prediction helper
batch_size = 64
val_loader = DataLoader(
    EventSequenceDataset(df, splits["val"], prep, config, mode="eval"),
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)
test_loader = DataLoader(
    EventSequenceDataset(df, splits["test"], prep, config, mode="eval"),
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)


def collect_predictions(model, loader):
    rows = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            entity_ids = batch["entity_id"]
            labels = batch["label"].cpu().numpy()
            moved = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            probs = torch.softmax(model(moved, mode="finetune")["logits"], dim=-1)[:, 1]
            for entity_id, label, proba in zip(entity_ids, labels, probs.cpu().numpy()):
                rows.append({GROUP_COL: entity_id, "target": int(label), "proba": float(proba)})
    return pd.DataFrame(rows)


In [ ]:
# Cell 6 - Low-label supervised encoder runs
run_rows = []
prediction_rows = []

for fraction in LABEL_FRACTIONS:
    for seed in LABEL_SAMPLING_SEEDS:
        set_seed(seed)
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        train_ids = sample_labeled_entities(splits["train"], labels_by_entity, fraction, seed)
        train_loader = DataLoader(
            EventSequenceDataset(df, train_ids, prep, config, mode="finetune"),
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_fn,
            num_workers=0,
        )

        run_id = f"supervised_f{int(fraction * 100):03d}_s{seed}"
        run_dir = CKPT_DIR / run_id
        run_dir.mkdir(parents=True, exist_ok=True)
        print(f"fraction={fraction:.2f} seed={seed} train_entities={len(train_ids)}")

        model = DMEEncoder(config, vocab_info).to(device)
        logger = MetricsLogger(str(LOG_DIR), run_id)
        best_ckpt = finetune(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            config=config,
            output_dir=str(run_dir),
            device=device,
            logger=logger,
            pretrained_checkpoint=None,
            frozen_encoder=False,
            label_fraction=1.0,
            vocab_info=vocab_info,
        )

        ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        metrics = evaluate_finetune(model, test_loader, num_classes=2, device=device)
        row = {"baseline": "supervised_encoder", "fraction": fraction, "seed": seed, **metrics}
        run_rows.append(row)
        print(row)

        preds = collect_predictions(model, test_loader)
        preds["fraction"] = fraction
        preds["seed"] = seed
        prediction_rows.append(preds)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

result_rows = append_mean_std_rows(run_rows, LABEL_FRACTIONS)
results_df = save_results(result_rows, "results_supervised_encoder")

predictions_df = pd.concat(prediction_rows, ignore_index=True)
pred_path = OUTPUT_DIR / "predictions_supervised_encoder.csv"
predictions_df.to_csv(pred_path, index=False)
print(f"Saved: {pred_path}")

try:
    display(results_df)
except NameError:
    print(results_df)
